# 🌸 Genshin Impact Hoyolab Wiki AI Agent

This notebook lets you:
1. **Scrape** Hoyolab Genshin Impact wiki pages and save them as local `.txt` files
2. **Ask Gemini** questions — it reads the saved files to answer you

---
## Setup

Install required libraries:

In [ ]:
# !pip install google-generativeai requests beautifulsoup4 playwright nest_asyncio
# !pip install playwright
# !playwright install chromium

## 1. Configuration

Set your **Gemini API Key** (get one free at https://aistudio.google.com/app/apikey):

In [1]:
import os

# --- CONFIGURE THESE ---
GEMINI_API_KEY = "gemini-API-key"   # <-- Paste your Gemini API key
WIKI_DATA_DIR  = "./wiki_data"                 # Folder where scraped .txt files are saved
# -----------------------

os.makedirs(WIKI_DATA_DIR, exist_ok=True)
print(f"✅ Wiki data will be saved to: {os.path.abspath(WIKI_DATA_DIR)}")

✅ Wiki data will be saved to: C:\Users\Anisa\OneDrive\Documents\Genshin AI Agent\wiki_data


In [2]:
import asyncio
import sys

# Force the Proactor event loop for Windows; fix Playwright asyncio issue on Windows
if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())


## 2. Wiki Scraper

Scrapes a Hoyolab Genshin Impact wiki page and saves the text locally.

**How to find wiki URLs:**
- Go to https://wiki.hoyolab.com/pc/genshin/home
- Navigate to any character, weapon, or game mechanic page
- Copy that URL and paste it below

In [3]:
from playwright.sync_api import sync_playwright
from pathlib import Path
import re
import time

def sanitize_filename(name: str) -> str:
    """Turn a page title or URL slug into a safe filename."""
    name = re.sub(r'[^\w\s-]', '', name).strip()
    name = re.sub(r'[\s-]+', '_', name)
    return name[:80]


def _scrape_wiki_page(url: str, label: str = None) -> str:
    """
    Scrapes a Hoyolab wiki page using a headless browser.
    Uses the sync Playwright API to avoid asyncio issues on Windows + Python 3.13.
    """
    print(f"🌐 Opening: {url}")

    with sync_playwright() as p:
        browser = p.chromium.launch(channel="chrome",headless=True)
        page = browser.new_page(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                       "AppleWebKit/537.36 (KHTML, like Gecko) "
                       "Chrome/120.0.0.0 Safari/537.36"
        )

        page.goto(url, wait_until="domcontentloaded", timeout=30000)

        # Wait longer for Hoyolab's JS to fully render
        page.wait_for_timeout(6000)

        # Wait specifically for Hoyolab wiki content
        try:
            page.wait_for_selector(
                ".wiki-entry-content, .entry-detail, .wiki-detail, "
                ".obc-tmpl, [class*='wiki'], [class*='entry'], [class*='detail']",
                timeout=20000
            )
            print("  ✅ Wiki content detected.")
        except Exception:
            print("  ⚠️  Could not detect wiki content — scraping whatever is loaded.")

        # Scroll down to trigger any lazy-loaded content
        page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
        page.wait_for_timeout(2000)  # Wait for lazy content to load after scroll


        # Get page title
        title = page.title()
        title = title.replace(" - HoYoLAB", "").replace(" | HoYoLAB", "").strip()

        # Extract visible text, removing clutter
        content = page.evaluate("""
            () => {
                const remove = ['nav', 'footer', 'script', 'style',
                                'header', 'aside', '[class*="ad"]',
                                '[class*="banner"]', '[class*="sidebar"]'];
                remove.forEach(sel => {
                    document.querySelectorAll(sel).forEach(el => el.remove());
                });
                return document.body.innerText;
            }
        """)

        browser.close()

    # Clean up whitespace
    lines = [line.strip() for line in content.splitlines()]
    lines = [l for l in lines if l]
    clean_text = "\n".join(lines)

    # Determine filename
    file_label = label if label else sanitize_filename(title if title else url.split("/")[-1])
    file_path = Path(WIKI_DATA_DIR) / f"{file_label}.txt"

    # Save with metadata header
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(f"SOURCE URL: {url}\n")
        f.write(f"PAGE TITLE: {title}\n")
        f.write(f"{'='*60}\n\n")
        f.write(clean_text)

    print(f"✅ Saved: {file_path}  ({len(clean_text):,} characters)")
    return str(file_path)


import threading

def scrape_wiki_page(url: str, label: str = None) -> str:
    """
    Runs the sync Playwright scraper in a separate thread to avoid
    Jupyter's background asyncio loop blocking it.
    """
    result = [None]
    error  = [None]

    def run():
        try:
            result[0] = _scrape_wiki_page(url, label)
        except Exception as e:
            error[0] = e

    thread = threading.Thread(target=run)
    thread.start()
    thread.join()

    if error[0]:
        raise error[0]
    return result[0]


print("✅ Scraper ready.")


✅ Scraper ready.


### 2a. Scrape Individual Pages

Paste any Hoyolab wiki URL below and run the cell:

In [ ]:
# --- Paste a Hoyolab wiki URL here ---
scrape_wiki_page(
    url   = "https://wiki.hoyolab.com/pc/genshin/entry/34",  # Example: Hu Tao
    label = "hu_tao"   # Optional: custom filename (no spaces)
)

### 2b. Batch-Scrape Multiple Pages

Add as many `(url, label)` pairs as you like:

In [ ]:
def find_max_entry_id(max_probe=20000, step=1000):
    """
    Probes by checking specific IDs rather than assuming
    consecutive entries — handles gaps in Hoyolab's entry IDs.
    """
    print("🔍 Probing for max entry ID...")
    
    last_valid = 1
    
    for probe_id in range(step, max_probe + step, step):
        url = f"https://sg-wiki-api.hoyolab.com/hoyowiki/genshin/wapi/entry_page?entry_page_id={probe_id}"
        try:
            resp = requests.get(url, headers={"x-rpc-language": "en-us"}, timeout=10)
            data = resp.json()
            
            if data.get("retcode") == 0 and data.get("data"):
                print(f"  ✅ Entry {probe_id} exists")
                last_valid = probe_id
            else:
                print(f"  ❌ Entry {probe_id} not found (retcode: {data.get('retcode')})")
        except Exception as e:
            print(f"  ⚠️ Error checking {probe_id}: {e}")
        
        time.sleep(0.2)  # Small delay to avoid rate limiting
    
    print(f"\n✅ Last confirmed valid entry: {last_valid}")
    print(f"   (There may be valid entries beyond this — increase max_probe if needed)")
    return last_valid

max_id = find_max_entry_id(max_probe=20000, step=1000)

In [8]:
# Set your base URL and the range of IDs you want
base_url = "https://wiki.hoyolab.com/pc/genshin/entry/"
start_id = 2782
end_id   = 10412  # Scrape entries 1 through 1000

for entry_id in range(start_id, end_id + 1):
    url = f"{base_url}{entry_id}"
    try:
        scrape_wiki_page(url, label=f"entry_{entry_id}")
    except Exception as e:
        print(f"❌ Failed entry {entry_id}: {e}")
    time.sleep(1)  # Small delay to avoid hammering the server

🌐 Opening: https://wiki.hoyolab.com/pc/genshin/entry/2782
  ✅ Wiki content detected.
✅ Saved: wiki_data\entry_2782.txt  (425 characters)
🌐 Opening: https://wiki.hoyolab.com/pc/genshin/entry/2783
  ✅ Wiki content detected.
✅ Saved: wiki_data\entry_2783.txt  (476 characters)
🌐 Opening: https://wiki.hoyolab.com/pc/genshin/entry/2784
  ✅ Wiki content detected.
✅ Saved: wiki_data\entry_2784.txt  (477 characters)
🌐 Opening: https://wiki.hoyolab.com/pc/genshin/entry/2785
  ✅ Wiki content detected.
✅ Saved: wiki_data\entry_2785.txt  (588 characters)
🌐 Opening: https://wiki.hoyolab.com/pc/genshin/entry/2786
  ✅ Wiki content detected.
✅ Saved: wiki_data\entry_2786.txt  (486 characters)
🌐 Opening: https://wiki.hoyolab.com/pc/genshin/entry/2787
  ✅ Wiki content detected.
✅ Saved: wiki_data\entry_2787.txt  (487 characters)
🌐 Opening: https://wiki.hoyolab.com/pc/genshin/entry/2788
  ✅ Wiki content detected.
✅ Saved: wiki_data\entry_2788.txt  (526 characters)
🌐 Opening: https://wiki.hoyolab.com/pc/ge

### 2c. View All Saved Files

In [4]:
saved_files = list(Path(WIKI_DATA_DIR).glob("*.txt"))

if not saved_files:
    print("⚠️  No files saved yet. Run the scraper cells above first.")
else:
    print(f"📂 {len(saved_files)} file(s) in {WIKI_DATA_DIR}:\n")
    for f in sorted(saved_files):
        size_kb = f.stat().st_size / 1024
        print(f"  • {f.name:<40} {size_kb:>6.1f} KB")

📂 10413 file(s) in ./wiki_data:

  • entry_1.txt                                15.5 KB
  • entry_10.txt                               31.5 KB
  • entry_100.txt                               0.3 KB
  • entry_1000.txt                              0.2 KB
  • entry_10000.txt                             0.2 KB
  • entry_10001.txt                             0.2 KB
  • entry_10002.txt                             0.2 KB
  • entry_10003.txt                             0.2 KB
  • entry_10004.txt                             0.2 KB
  • entry_10005.txt                             0.2 KB
  • entry_10006.txt                             0.2 KB
  • entry_10007.txt                             0.2 KB
  • entry_10008.txt                             0.2 KB
  • entry_10009.txt                             0.2 KB
  • entry_1001.txt                              0.2 KB
  • entry_10010.txt                             0.2 KB
  • entry_10011.txt                             0.2 KB
  • entry_10012.txt             

---
## 3. Gemini Agent

The agent loads the saved `.txt` files and uses Gemini to answer questions.

In [7]:
import google.generativeai as genai
from pathlib import Path

# Configure Gemini
genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel(
    model_name="gemini-2.5-flash",   # Fast and free-tier friendly
    system_instruction=(
        "You are a knowledgeable Genshin Impact assistant. "
        "You will be provided with text extracted from the Hoyolab Genshin Impact wiki. "
        "Use ONLY the provided wiki content to answer the user's question. "
        "If the information isn't in the provided content, say so clearly. "
        "Be concise, accurate, and helpful."
    )
)


def load_wiki_files(filenames: list = None) -> str:
    """
    Load wiki text files into a single context string.
    
    - filenames=None  → loads ALL .txt files in WIKI_DATA_DIR
    - filenames=['hu_tao', 'amber']  → loads only those files
    """
    data_path = Path(WIKI_DATA_DIR)
    
    if filenames:
        files = [data_path / (f if f.endswith(".txt") else f + ".txt") for f in filenames]
    else:
        files = sorted(data_path.glob("*.txt"))
    
    if not files:
        return ""
    
    parts = []
    for fp in files:
        if fp.exists():
            text = fp.read_text(encoding="utf-8")
            parts.append(f"\n\n{'='*60}\nFILE: {fp.name}\n{'='*60}\n{text}")
        else:
            print(f"  ⚠️  File not found: {fp}")
    
    return "\n".join(parts)


def ask(question: str, files: list = None, verbose: bool = False) -> str:
    """
    Ask Gemini a question using the loaded wiki files as context.
    
    Args:
        question : Your question about Genshin Impact
        files    : List of file labels to load (None = load all)
        verbose  : If True, show which files were loaded
    
    Returns:
        Gemini's answer as a string
    """
    wiki_context = load_wiki_files(files)
    
    if not wiki_context:
        return ("⚠️  No wiki files found. "
                "Please run the scraper cells above to save some wiki pages first.")
    
    loaded = files if files else [f.name for f in Path(WIKI_DATA_DIR).glob("*.txt")]
    if verbose:
        print(f"📚 Loaded {len(loaded)} file(s): {', '.join(str(l) for l in loaded)}")
        print(f"📝 Context size: {len(wiki_context):,} characters\n")
    
    prompt = (
        f"Here is content from the Genshin Impact Hoyolab wiki:\n"
        f"{wiki_context}\n\n"
        f"Based on the wiki content above, please answer this question:\n"
        f"{question}"
    )
    
    response = model.generate_content(prompt)
    return response.text


print("✅ Gemini agent ready. Use ask('your question') to query the wiki.")

C:\Users\Anisa\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Anisa\AppData\Local\Temp\ipykernel_23504\1994462312.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


NameError: name 'GEMINI_API_KEY' is not defined

In [10]:
# Check available models and their limits
for model in genai.list_models():
    if "gemini-2.5-flash" in model.name:
        print(f"Model: {model.name}")
        print(f"Input token limit:  {model.input_token_limit:,}")
        print(f"Output token limit: {model.output_token_limit:,}")
        print()


# Count tokens for your loaded wiki files
wiki_context = load_wiki_files()  # Load all saved files

model = genai.GenerativeModel("gemini-2.5-flash")
token_count = model.count_tokens(wiki_context)

print(f"📄 Characters in context: {len(wiki_context):,}")
print(f"🔢 Tokens in context:     {token_count.total_tokens:,}")
print()

# Free tier limits for gemini-2.0-flash
FREE_TIER_LIMIT = 1_000_000  # tokens per minute
print(f"Free tier limit:  {FREE_TIER_LIMIT:,} tokens/minute")
print(f"Your usage:       {token_count.total_tokens:,} tokens")
print(f"Remaining:        {FREE_TIER_LIMIT - token_count.total_tokens:,} tokens")

Model: models/gemini-2.5-flash
Input token limit:  1,048,576
Output token limit: 65,536

Model: models/gemini-2.5-flash-preview-tts
Input token limit:  8,192
Output token limit: 16,384

Model: models/gemini-2.5-flash-lite
Input token limit:  1,048,576
Output token limit: 65,536

Model: models/gemini-2.5-flash-image
Input token limit:  32,768
Output token limit: 32,768

Model: models/gemini-2.5-flash-native-audio-latest
Input token limit:  131,072
Output token limit: 8,192

Model: models/gemini-2.5-flash-native-audio-preview-09-2025
Input token limit:  131,072
Output token limit: 8,192

Model: models/gemini-2.5-flash-native-audio-preview-12-2025
Input token limit:  131,072
Output token limit: 8,192

📄 Characters in context: 16,710,459
🔢 Tokens in context:     4,357,046

Free tier limit:  1,000,000 tokens/minute
Your usage:       4,357,046 tokens
Remaining:        -3,357,046 tokens


---
## 4. Ask Questions!

### Simple query (searches all saved files)

In [9]:
answer = ask("What element does Hu Tao use and what is her role in a team?")
print(answer)

⚠️  Context truncated to 12,000 characters for local model
❌ Error: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=120)


### Target specific files (faster, more focused)

In [10]:
answer = ask(
    question = "What are Hu Tao's best weapons?",
    files    = ["hu_tao"],   # Only load hu_tao.txt
    verbose  = True
)
print(answer)

📚 Loaded 1 file(s): hu_tao
📝 Context size: 34,828 characters

⚠️  Context truncated to 12,000 characters for local model
❌ Error: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=120)


### Interactive chat loop

Keep asking questions until you type `quit`:

In [11]:
print("🤖 Genshin Wiki Agent — type 'quit' to exit\n")
print(f"📂 Loaded files: {[f.name for f in Path(WIKI_DATA_DIR).glob('*.txt')]}\n")

while True:
    question = input("You: ").strip()
    if not question:
        continue
    if question.lower() in ("quit", "exit", "q"):
        print("👋 Bye!")
        break
    
    print("\n🌸 Gemini:")
    answer = ask(question)
    print(answer)
    print("\n" + "-"*50 + "\n")

🤖 Genshin Wiki Agent — type 'quit' to exit

📂 Loaded files: ['entry_1.txt', 'entry_10.txt', 'entry_100.txt', 'entry_1000.txt', 'entry_10000.txt', 'entry_10001.txt', 'entry_10002.txt', 'entry_10003.txt', 'entry_10004.txt', 'entry_10005.txt', 'entry_10006.txt', 'entry_10007.txt', 'entry_10008.txt', 'entry_10009.txt', 'entry_1001.txt', 'entry_10010.txt', 'entry_10011.txt', 'entry_10012.txt', 'entry_10013.txt', 'entry_10014.txt', 'entry_10015.txt', 'entry_10016.txt', 'entry_10017.txt', 'entry_10018.txt', 'entry_10019.txt', 'entry_1002.txt', 'entry_10020.txt', 'entry_10021.txt', 'entry_10022.txt', 'entry_10023.txt', 'entry_10024.txt', 'entry_10025.txt', 'entry_10026.txt', 'entry_10027.txt', 'entry_10028.txt', 'entry_10029.txt', 'entry_1003.txt', 'entry_10030.txt', 'entry_10031.txt', 'entry_10032.txt', 'entry_10033.txt', 'entry_10034.txt', 'entry_10035.txt', 'entry_10036.txt', 'entry_10037.txt', 'entry_10038.txt', 'entry_10039.txt', 'entry_1004.txt', 'entry_10040.txt', 'entry_10041.txt', 'en

You:  尼可应该用什么武器



🌸 Gemini:


ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_paid_tier_input_token_count, limit: 1000000, model: gemini-2.5-flash
Please retry in 26.530858625s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_paid_tier_input_token_count"
  quota_id: "GenerateContentPaidTierInputTokensPerModelPerMinute"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 1000000
}
, retry_delay {
  seconds: 26
}
]

---
## 5. Utilities

### Preview a saved file

In [ ]:
def preview_file(label: str, lines: int = 50):
    fp = Path(WIKI_DATA_DIR) / f"{label}.txt"
    if not fp.exists():
        print(f"File not found: {fp}")
        return
    content = fp.read_text(encoding="utf-8").splitlines()
    print(f"\n--- {fp.name} (first {lines} lines) ---\n")
    print("\n".join(content[:lines]))
    if len(content) > lines:
        print(f"\n... ({len(content) - lines} more lines)")

preview_file("hu_tao")

### Delete a saved file

In [ ]:
def delete_file(label: str):
    fp = Path(WIKI_DATA_DIR) / f"{label}.txt"
    if fp.exists():
        fp.unlink()
        print(f"🗑️  Deleted: {fp.name}")
    else:
        print(f"File not found: {fp}")

# delete_file("hu_tao")   # Uncomment to delete

---
## Quick Reference

| Task | Code |
|------|------|
| Scrape a page | `scrape_wiki_page(url, label)` |
| List saved files | `list(Path(WIKI_DATA_DIR).glob('*.txt'))` |
| Ask using all files | `ask('your question')` |
| Ask using specific files | `ask('question', files=['hu_tao', 'amber'])` |
| Preview a file | `preview_file('hu_tao')` |
| Delete a file | `delete_file('hu_tao')` |

**Good Hoyolab wiki pages to scrape:**
- Characters: `https://wiki.hoyolab.com/pc/genshin/entry/{id}`
- Browse the wiki at: https://wiki.hoyolab.com/pc/genshin/home